# FIT TRADIZIONALE con multi-start Monte Carlo


---

Fit di tutti gli spettri usando le classi [PoissonNoise](/ml_odmr/PoissonNoise.py) e [FitMonteCarlo](/ml_odmr/FitMonteCarlo.py).

In [2]:
import h5py
import numpy as np
from numba import njit
from scipy.optimize import curve_fit
from joblib import Parallel, delayed
import time
from ml_odmr.PoissonNoise import PoissonNoise
from ml_odmr.FitMonteCarlo import FitMonteCarlo

# Lettura degli spettri grezzi e asse frequenze
file_path = "../data/10k/odmr_easyspin_raw_10000.mat"
with h5py.File(file_path, "r") as f:
    spectra_data = np.array(f["spectra_data"]).T
    mw_freqs = np.array(f["mw_freqs"]).squeeze()
    labels_c = np.array(f["labels_c"]).squeeze()
    labels_s = np.array(f["labels_s"]).squeeze()

f_ghz = np.asarray(mw_freqs, dtype=np.float64)
N_tot_samples = len(spectra_data)

# Inizializziamo i generatori
noise_gen = PoissonNoise(seed=42)
fitter = FitMonteCarlo(M=20)

# Esempio: fittiamo tutti gli spettri del test set (es. 1000 spettri) 
# oppure tutti i 10.000 cambiando il range.
n_to_fit = 1000 
print(f"Avvio del fit Monte Carlo su {n_to_fit} spettri...")
t_start = time.time()

def worker(idx):
    # Generiamo il segnale rumoroso con budget casuale di fotoni o fisso
    y_raw = spectra_data[idx]
    ideal = noise_gen.ideal_signal(y_raw, contrast=0.08)
    # n_tot variabile tra 3e5 e 6e6 per coprire tutto l'intervallo di SNR
    n_photons = np.random.uniform(3.0e5, 6.0e6)
    noisy = noise_gen.add_noise(ideal, n_tot=n_photons, seed_modifier=idx)
    
    c_norm, s_norm, c_phys, s_phys, popt, chi2 = fitter.fit_single(f_ghz, noisy)
    return c_norm, s_norm, c_phys, s_phys, chi2, n_photons

# Esecuzione in parallelo su tutti i core CPU disponibili
results = Parallel(n_jobs=-1, verbose=10)(
    delayed(worker)(i) for i in range(n_to_fit)
)

t_tot = time.time() - t_start
print(f"\nFit completato in {t_tot:.1f} secondi! ({t_tot/n_to_fit*1e3:.2f} ms/spettro)")

# Convertiamo i risultati in matrici NumPy
results_arr = np.array([[r[0], r[1], r[2], r[3], r[4], r[5]] for r in results])
fit_c_norm = results_arr[:, 0]
fit_s_norm = results_arr[:, 1]
fit_c_phys = results_arr[:, 2]
fit_s_phys = results_arr[:, 3]
fit_chi2   = results_arr[:, 4]
photons    = results_arr[:, 5]

# Calcolo errori rispetto alla Ground Truth
true_c = labels_c[:n_to_fit]
true_s = labels_s[:n_to_fit]

valid = ~np.isnan(fit_c_norm)
mae_c = np.mean(np.abs(fit_c_norm[valid] - true_c[valid]))
mae_s = np.mean(np.abs(fit_s_norm[valid] - true_s[valid]))
failures = np.sum(~valid)

print(f"Spettri falliti/non convergenti: {failures}/{n_to_fit}")
print(f"MAE medio normalizzato su c: {mae_c:.4f}")
print(f"MAE medio normalizzato su s: {mae_s:.4f}")

Avvio del fit Monte Carlo su 1000 spettri...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    2.9s
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    4.1s
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:    5.0s
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed:    6.0s
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    7.2s
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:    8.2s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:   10.1s
[Parallel(n_jobs=-1)]: Done  69 tasks      | elapsed:   11.0s
[Parallel(n_jobs=-1)]: Done  82 tasks      | elapsed:   12.3s
[Parallel(n_jobs=-1)]: Done  97 tasks      | elapsed:   13.7s
[Parallel(n_jobs=-1)]: Done 112 tasks      | elapsed:   15.2s
[Parallel(n_jobs=-1)]: Done 129 tasks      | elapsed:   17.2s
[Parallel(n_jobs=-1)]: Done 146 tasks      | elapsed:   19.4s
[Parallel(n_jobs=-1)]: Done 165 tasks      | elapsed:   21.7s
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed:   


Fit completato in 147.2 secondi! (147.18 ms/spettro)
Spettri falliti/non convergenti: 0/1000
MAE medio normalizzato su c: 0.0017
MAE medio normalizzato su s: 0.0020


[Parallel(n_jobs=-1)]: Done 1000 out of 1000 | elapsed:  2.5min finished
